**Starting Local Redis**: in Terminal

`docker ps` --> check to see if container is running

`docker start redis-project` --> if container is stopped

`docker exec redis-project redis-cli PING` --> connect to CLI and test connection

Other useful commands:

`docker stop redis-project` --> stop container

`docker restart redis-project` --> restart container

`FLUSHDB` and `DBSIZE` --> wipe data and make sure size is 0

In [4]:
import pandas as pd
import redis

In [5]:
# Connect to Redis
r = redis.Redis(
    host = "localhost",
    port = 6379,
    decode_responses = True
)

In [15]:
DATA = "/Users/simontao/Desktop/Fall 2026/DATS6102/redis_project/data/202609-cbs-tripdata-preprocessed.csv" # path to the CSV on your machine

In [16]:
CHUNK_SIZE = 5000

for chunk in pd.read_csv(
    DATA, 
    chunksize=5000,
    dtype = {"ride_id": "string", "start_station_id": "string", "end_station_id": "string"}
    ):
    
    # Pipeline setup
    chunk = chunk.fillna("")
    pipeline = r.pipeline(transaction=False)

    # Chunk processing
    for row in chunk.to_dict("records"):
        ride_id = row["ride_id"]
        start_id = row["start_station_id"]
        end_id = row["end_station_id"]

        # Individual ride hashes
        pipeline.hset(
            f"ride:{ride_id}",
            mapping=row
        )

        # Skip station operations if either station is missing
        if not start_id or not end_id:
            continue

        # Individual station hashes
        pipeline.hset(
            f"station:{start_id}", # Starting station
            mapping={
                "name": row["start_station_name"],
                "latitude": row["start_lat"],
                "longitude": row["start_lng"]
            }
        )
        
        pipeline.hset(
            f"station:{end_id}", # Ending station
            mapping={
                "name": row["end_station_name"],
                "latitude": row["end_lat"],
                "longitude": row["end_lng"]
            }
        )

        # Set: Unique destinations from one origin
        pipeline.sadd(
            f"station:{start_id}:destinations",
            end_id
        )

        # Sorted set: Departure ranking
        pipeline.zincrby(
            "stations:departures",
            1,
            start_id
        )

        # Sorted set: Arrival ranking
        pipeline.zincrby(
            "stations:arrivals",
            1,
            end_id
        )

        # Sorted set: Route rankings
        route = f"{start_id}->{end_id}"

        pipeline.zincrby(
            "routes:popular",
            1,
            route
        )

    pipeline.execute()

    print(f"Loaded {len(chunk):,} rides")

print("Finished!")

Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000 rides
Loaded 5,000

# Redis Queries

**Hash: Ride information**
- Get all information about one ride: `HGETALL ride:9FF60D31E16B6637`
- From that one ride, get the starting station name: `HGET ride:9FF60D31E16B6637 start_station_name`

**Hash: Station information**
- Get all information about one station: `HGETALL station:31623`
- Get the name from that one station: `HGET station:31623 name`

**Set: Get all destinations from one origin station**
- List all stations: `SMEMBERS station:31623:destinations`
- Count all stations: `SCARD station:31623:destinations`

**Sorted Set: Top 10 stations that riders arrive at**
- Top 10 arrival stations: `ZREVRANGE stations:arrivals 0 9 WITHSCORES`
- Number of arrivals at a particular station: `ZSCORE stations:arrivals <station_id>`

**Sorted Set: Top 10 stations that riders depart from**
- Top 10 departure stations: `ZREVRANGE stations:departures 0 9 WITHSCORES`
- Number of departures at a particular station: `ZSCORE stations:departures <station_id>`

**Sorted Set: Top 10 routes taken by riders**
- Top 10 most popular routes: `ZREVRANGE routes:popular 0 9 WITHSCORES`

### SQL Comparison

SQL Tables:

**rides:** each row is a ride ID and its associated characteristics

**stations**: each row is a station ID and its associated characteristics

**routes**: each row can have ride ID, start station ID, and end station ID

In [ ]:
-- SQL for Hash: Ride information
SELECT * 
FROM rides
WHERE ride_id = "9FF60D31E16B6637";

SELECT start_station_name
FROM rides
WHERE ride_id = "9FF60D31E16B6637";

In [ ]:
-- SQL for Hash: Station Information
SELECT *
FROM stations
WHERE station_id = "31623";

SELECT name
FROM stations
WHERE station_id = "31623";

In [ ]:
-- SQL for Set: Get all destinations from one origin station
SELECT DISTINCT end_station_id
FROM routes
WHERE start_station_id = "31623";

SELECT COUNT(DISTINCT end_station_id) AS destination_count
FROM routes
WHERE start_station_id = '31623';

In [ ]:
-- Sorted Set: Top 10 stations that riders arrive at
SELECT end_station_id, COUNT(*) AS arrival_count
FROM routes
GROUP BY end_station_id
ORDER BY arrival_count DESC
LIMIT 10;

SELECT COUNT(*) AS arrival_count
FROM routes
WHERE end_station_id = '31623';

In [ ]:
-- Sorted Set: Top 10 stations that riders depart from
SELECT start_station_id, COUNT(*) AS departure_count
FROM routes
GROUP BY start_station_id
ORDER BY departure_count DESC
LIMIT 10;

SELECT COUNT(*) AS departure_count
FROM routes
WHERE start_station_id = '31623';

In [ ]:
-- Sorted Set: Top 10 routes taken by riders
SELECT
    start_station_id,
    end_station_id,
    COUNT(*) AS route_count
FROM routes
GROUP BY start_station_id, end_station_id
ORDER BY route_count DESC
LIMIT 10;